In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[1]
sys.path.append(str(PROJECT_ROOT))

In [2]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import random
import time
from scipy.stats import mode
from loaders._load_vn30_multi_class_tree import preprocess, VN30, TARGETS
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score

In [3]:
class EnhancedDecisionTreeEnsemble:
    def __init__(self, n_estimators=10, pool_multiplier=2, random_state=42):
        """
        Enhanced Decision Tree Ensemble với thuật toán chọn cây độc lập
        
        Parameters:
        - n_estimators: số cây cuối cùng muốn có trong ensemble
        - pool_multiplier: hệ số nhân để tạo pool (pool_size = n_estimators * pool_multiplier)
        - random_state: seed cho reproducibility
        """
        self.n_estimators = n_estimators
        self.pool_size = n_estimators * pool_multiplier
        self.random_state = random_state
        self.selected_trees = []
        self.tree_pool = []
        
    def _generate_random_params(self):
        """Tạo tham số ngẫu nhiên cho Decision Tree"""
        return {
            'max_depth': random.randint(5, 10),
            'random_state': random.randint(0, 1000)
        }
    
    def _create_tree_pool(self, X_train, y_train):
        """Tạo pool các Decision Tree với tham số ngẫu nhiên và train trên các segment khác nhau"""
        random.seed(self.random_state)
        np.random.seed(self.random_state)
        
        n_samples = len(X_train)
        segment_size = max(int(0.7 * n_samples), 100)  # Mỗi segment chiếm 70% dữ liệu, tối thiểu 100 samples
        
        self.tree_pool = []
        for i in range(self.pool_size):
            # Chọn điểm bắt đầu ngẫu nhiên cho segment
            max_start = n_samples - segment_size
            start_idx = random.randint(0, max_start) if max_start > 0 else 0
            end_idx = start_idx + segment_size
            
            # Cắt segment từ dữ liệu train
            X_segment = X_train[start_idx:end_idx]
            y_segment = y_train[start_idx:end_idx]
            
            params = self._generate_random_params()
            tree = DecisionTreeClassifier(**params)
            tree.fit(X_segment, y_segment)
            self.tree_pool.append(tree)
            
    def _calculate_covariance_matrix(self, X_val, y_val):
        """Tính ma trận covariance của predictions trên validation set"""
        predictions = []
        for tree in self.tree_pool:
            pred = tree.predict(X_val)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        # Tính covariance matrix của predictions
        cov_matrix = np.cov(predictions)
        return cov_matrix, predictions
    
    def _select_independent_trees(self, cov_matrix, predictions, y_val):
        """Chọn các cây có covariance tối thiểu và accuracy cao"""
        n_trees = len(self.tree_pool)
        
        # Tính accuracy của từng cây
        accuracies = []
        for i in range(n_trees):
            acc = balanced_accuracy_score(y_val, predictions[i])
            accuracies.append(acc)
        
        # Thuật toán greedy để chọn cây
        selected_indices = []
        remaining_indices = list(range(n_trees))
        
        # Chọn cây đầu tiên có accuracy cao nhất
        best_idx = np.argmax(accuracies)
        selected_indices.append(best_idx)
        remaining_indices.remove(best_idx)
        
        # Chọn các cây tiếp theo
        for _ in range(self.n_estimators - 1):
            if not remaining_indices:
                break
                
            best_score = float('inf')
            best_candidate = None
            
            for candidate in remaining_indices:
                # Tính tổng covariance với các cây đã chọn
                total_cov = 0
                for selected in selected_indices:
                    total_cov += abs(cov_matrix[candidate, selected])
                
                # Score = covariance / accuracy (muốn minimize covariance, maximize accuracy)
                score = total_cov / (accuracies[candidate] + 1e-6)
                
                if score < best_score:
                    best_score = score
                    best_candidate = candidate
            
            if best_candidate is not None:
                selected_indices.append(best_candidate)
                remaining_indices.remove(best_candidate)
        
        return selected_indices
    
    def fit(self, X_train, y_train, X_val, y_val):
        """Train ensemble"""
        self._create_tree_pool(X_train, y_train)
        
        cov_matrix, predictions = self._calculate_covariance_matrix(X_val, y_val)
        
        selected_indices = self._select_independent_trees(cov_matrix, predictions, y_val)
        
        self.selected_trees = [self.tree_pool[i] for i in selected_indices]
            
        return self
    
    def predict(self, X):
        """Dự đoán bằng voting"""
        if not self.selected_trees:
            raise ValueError("Model chưa được train!")
            
        predictions = []
        for tree in self.selected_trees:
            pred = tree.predict(X)
            predictions.append(pred)
        
        predictions = np.array(predictions)
        # Voting: lấy class xuất hiện nhiều nhất
        final_predictions = []
        for i in range(predictions.shape[1]):
            votes = predictions[:, i]
            most_common = mode(votes, keepdims=False)[0]
            final_predictions.append(most_common)
            
        return np.array(final_predictions)

In [4]:
mean_accuracy = []
for symbol in VN30:
    print(f"\n{'='*20} {symbol} {'='*20}")
    data = preprocess(symbol, lag=30, lag_label=True, val=0.2)

    X_train, Y_train = data['train']
    X_val, Y_val = data['val']
    X_test, Y_test = data['test']
    classes = data['classes']
    
    # Enhanced Ensemble
    enhanced_model = EnhancedDecisionTreeEnsemble(
        n_estimators=50,  # số cây cuối cùng
        pool_multiplier=2,  # tạo pool gấp hai
        random_state=0
    )
    enhanced_model.fit(X_train, Y_train, X_val, Y_val)
    Y_pred_enhanced = enhanced_model.predict(X_test)
    accuracy_enhanced = balanced_accuracy_score(Y_test, Y_pred_enhanced)
    mean_accuracy.append(accuracy_enhanced)

    print(f"\nResults for {symbol}:")
    print(f"Enhanced Ensemble:    {accuracy_enhanced:.4f}")

    time.sleep(2)


print(f"\n{'='*60}")
print("FINAL RESULTS:")
print(f"Mean Enhanced Accuracy:  {np.mean(mean_accuracy):.4f}")
print(f"Standard deviation of accuracy: {np.std(mean_accuracy):.4f}")
print(f"{'='*60}")


==================== ACB ====================

Results for ACB:
Enhanced Ensemble:    0.2182

==================== BCM ====================

Results for BCM:
Enhanced Ensemble:    0.2235

==================== BID ====================

Results for BID:
Enhanced Ensemble:    0.2068

==================== BVH ====================

Results for BVH:
Enhanced Ensemble:    0.2025

==================== CTG ====================

Results for CTG:
Enhanced Ensemble:    0.1887

==================== FPT ====================

Results for FPT:
Enhanced Ensemble:    0.2167

==================== GAS ====================

Results for GAS:
Enhanced Ensemble:    0.2003

==================== GVR ====================

Results for GVR:
Enhanced Ensemble:    0.2569

==================== HDB ====================

Results for HDB:
Enhanced Ensemble:    0.2000

==================== HPG ====================

Results for HPG:
Enhanced Ensemble:    0.2000

==================== LPB ====================

Results for 